# RT-DETR Hyperparameter Trials — Garbage & Bin Detection

This notebook covers the final set of hyperparameter experiments used to train and evaluate RT-DETR on a merged Moroccan street dataset containing `garbage` and `bin` classes.

**Pipeline overview:**
1. Environment setup
2. Dataset merge & deduplication
3. Data loading & exploration
4. Stratified train / val / test split
5. YAML config
6. Experiment definitions
7. Training loop


## 1. Environment Setup

In [ ]:
!pip install -q ultralytics
import ultralytics; ultralytics.checks()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Dataset Merge & Deduplication

Merges up to four source directories into one, keeping only annotated images and skipping exact content duplicates (MD5 hash) and filename collisions.

In [ ]:
# ── Merge two dataset directories — keep only images with annotations, no duplicates ──
import os, shutil, hashlib
from pathlib import Path

# ── EDIT THESE ───────────────────────────────────────────────────────────────
SOURCE_A = '/content/drive/MyDrive/new/dataset_merged/train/'   # first  source: must contain images/ and labels/
SOURCE_B = '/content/drive/MyDrive/Shared/Data/'   # second source: must contain images/ and labels/
SOURCE_C = '/content/drive/MyDrive/PA_Project_data/'
SOURCE_D = '/content/drive/MyDrive/Shared/Folder_a/'
MERGED   = '/content/drive/MyDrive/DataMerged_AISIN' # output: will be created
# ─────────────────────────────────────────────────────────────────────────────

IMG_EXTS = {'.jpg', '.jpeg', '.png'}

out_imgs = Path(MERGED) / 'images'
out_lbls = Path(MERGED) / 'labels'
out_imgs.mkdir(parents=True, exist_ok=True)
out_lbls.mkdir(parents=True, exist_ok=True)

def md5(path):
    """Fast content hash to detect true duplicates regardless of filename."""
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()

seen_hashes = set()   # catches identical image files with different names
seen_stems  = set()   # catches same filename from both sources
copied = skipped_no_label = skipped_dup = 0

for source in [SOURCE_A, SOURCE_B, SOURCE_C, SOURCE_D]:
    img_dir = Path(source) / 'images'
    lbl_dir = Path(source) / 'labels'

    if not img_dir.exists():
        print(f'⚠️  images/ not found in {source} — skipping')
        continue

    for img_path in sorted(img_dir.iterdir()):
        if img_path.suffix.lower() not in IMG_EXTS:
            continue

        lbl_path = lbl_dir / (img_path.stem + '.txt')

        # 1 — skip if no matching annotation exists
        if not lbl_path.exists():
            skipped_no_label += 1
            continue

        # 2 — skip exact content duplicates
        h = md5(img_path)
        if h in seen_hashes:
            skipped_dup += 1
            continue

        # 3 — resolve filename collision: if stem already taken, append source tag
        dest_stem = img_path.stem
        if dest_stem in seen_stems:
            tag = Path(source).name  # e.g. 'DatasetA'
            dest_stem = f'{img_path.stem}_{tag}'

        seen_hashes.add(h)
        seen_stems.add(dest_stem)

        shutil.copy(img_path, out_imgs / (dest_stem + img_path.suffix))
        shutil.copy(lbl_path, out_lbls / (dest_stem + '.txt'))
        copied += 1

print(f'\n✅ Merge complete')
print(f'   Copied          : {copied} image+label pairs')
print(f'   Skipped (no lbl): {skipped_no_label}')
print(f'   Skipped (dup)   : {skipped_dup}')
print(f'   Output          : {MERGED}')

# Point RAW_DIR at the merged output so the rest of the notebook uses it
RAW_DIR = MERGED
IMG_DIR = str(out_imgs)
LBL_DIR = str(out_lbls)
print(f'\nRAW_DIR is now set to: {RAW_DIR}')


### 2a. Drive Folder Inspection *(optional)*

Scan Drive to confirm the merged `images/` directories are in the expected locations.

In [ ]:
from pathlib import Path

base = Path("/content/drive")

for p in base.rglob("*"):
    if p.is_dir() and "images" in p.name.lower():
        print(p)

## 3. Data Loading & Class Distribution

Load all images from the merged dataset, count per-class instances, and compute inverse-frequency class weights for potential use in loss.

In [ ]:
import os, shutil, random
from pathlib import Path
from collections import Counter
import yaml
import numpy as np
import matplotlib.pyplot as plt
import cv2

# ── EDIT THIS ────────────────────────────────────────────
RAW_DIR = '/content/drive/MyDrive/new/dataset_merged/train'
# ─────────────────────────────────────────────────────────

IMG_DIR = os.path.join(RAW_DIR, 'images')
LBL_DIR = os.path.join(RAW_DIR, 'labels')
CLASS_NAMES = ['garbage', 'bin']

images = sorted([f for f in os.listdir(IMG_DIR)
                 if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
print(f'Total images: {len(images)}')

In [ ]:
# ── Class distribution — check imbalance before training ─
label_counts = Counter()
for img in images:
    lbl = os.path.join(LBL_DIR, os.path.splitext(img)[0] + '.txt')
    if os.path.exists(lbl):
        for line in open(lbl):
            p = line.strip().split()
            if p: label_counts[int(p[0])] += 1

counts = [label_counts[i] for i in range(len(CLASS_NAMES))]
print('Instance counts:', dict(zip(CLASS_NAMES, counts)))

# Class weights for loss — inverse frequency, useful when bin >> garbage or vice versa
total = sum(counts)
CLASS_WEIGHTS = [total / (len(counts) * c) for c in counts]
print('Class weights (inverse freq):', dict(zip(CLASS_NAMES, [round(w,3) for w in CLASS_WEIGHTS])))

plt.bar(CLASS_NAMES, counts, color=['#e74c3c','#3498db'])
plt.title('Instance distribution'); plt.ylabel('Count'); plt.tight_layout(); plt.show()

In [ ]:
import random
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

# === CONFIG ===
images_dir = Path(os.path.join(RAW_DIR, 'images'))
labels_dir = Path(os.path.join(RAW_DIR, 'labels'))


class_names = ["garbage", "bin"]  # <-- MUST match your final mapping

num_samples = 5  # how many images to display

# === GET IMAGES ===
image_files = list(images_dir.glob("*.*"))
samples = random.sample(image_files, min(num_samples, len(image_files)))

# === FUNCTION TO DRAW BOXES ===
def draw_yolo_boxes(img, label_path):
    h, w = img.shape[:2]

    if not label_path.exists():
        return img

    with open(label_path, "r") as f:
        lines = f.readlines()

    for line in lines:
        parts = line.strip().split()
        if len(parts) != 5:
            continue

        class_id, x, y, bw, bh = map(float, parts)

        # Convert YOLO → pixel coords
        x1 = int((x - bw/2) * w)
        y1 = int((y - bh/2) * h)
        x2 = int((x + bw/2) * w)
        y2 = int((y + bh/2) * h)

        label = class_names[int(class_id)]

        # === BOX ===
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 3)

        # === TEXT SETTINGS (scaled with image size) ===
        font_scale = max(0.6, min(w, h) / 600)
        thickness = 2

        (text_w, text_h), _ = cv2.getTextSize(
            label, cv2.FONT_HERSHEY_SIMPLEX, font_scale, thickness
        )

        # Background rectangle for text
        cv2.rectangle(
            img,
            (x1, y1 - text_h - 10),
            (x1 + text_w, y1),
            (0, 255, 0),
            -1
        )

        # Put text (black for contrast)
        cv2.putText(
            img,
            label,
            (x1, y1 - 5),
            cv2.FONT_HERSHEY_SIMPLEX,
            font_scale,
            (0, 0, 0),
            thickness,
            cv2.LINE_AA
        )

    return img

# === DISPLAY ===
plt.figure(figsize=(30, 10))

for i, img_path in enumerate(samples):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    label_path = labels_dir / (img_path.stem + ".txt")

    img = draw_yolo_boxes(img, label_path)

    plt.subplot(1, len(samples), i+1)
    plt.imshow(img)
    plt.title(img_path.name)
    plt.axis("off")

plt.tight_layout()
plt.show()

## 4. Stratified Train / Val / Test Split

Buckets images by dominant class label before splitting (70 / 20 / 10) so that each subset preserves the overall class ratio.

In [ ]:
# ── Stratified split — preserves class balance across splits ──
# Group images by dominant class so each split gets a fair share
random.seed(42)
np.random.seed(42)

DATASET_DIR = Path('/content/dataset')
for split in ['train', 'val', 'test']:
    (DATASET_DIR / split / 'images').mkdir(parents=True, exist_ok=True)
    (DATASET_DIR / split / 'labels').mkdir(parents=True, exist_ok=True)

# Bucket images by dominant class label
buckets = {i: [] for i in range(len(CLASS_NAMES))}
no_label = []
for img in images:
    lbl = os.path.join(LBL_DIR, os.path.splitext(img)[0] + '.txt')
    if not os.path.exists(lbl):
        no_label.append(img); continue
    cls_ids = [int(l.split()[0]) for l in open(lbl) if l.strip()]
    dominant = Counter(cls_ids).most_common(1)[0][0] if cls_ids else 0
    buckets[dominant].append(img)

split_map = []
for cls_id, imgs in buckets.items():
    random.shuffle(imgs)
    n = len(imgs)
    n_tr = int(n * 0.70); n_v = int(n * 0.20)
    split_map += [('train', i) for i in imgs[:n_tr]]
    split_map += [('val',   i) for i in imgs[n_tr:n_tr+n_v]]
    split_map += [('test',  i) for i in imgs[n_tr+n_v:]]

for split, img_name in split_map:
    stem = os.path.splitext(img_name)[0]
    shutil.copy(os.path.join(IMG_DIR, img_name), DATASET_DIR / split / 'images' / img_name)
    shutil.copy(os.path.join(LBL_DIR, stem+'.txt'), DATASET_DIR / split / 'labels' / (stem+'.txt'))

counts_split = Counter(s for s, _ in split_map)
for s in ['train','val','test']:
    print(f'{s:5s}: {counts_split[s]:4d} images ({counts_split[s]/len(split_map)*100:.1f}%)')

## 5. Dataset YAML

Writes the Ultralytics-compatible `dataset.yaml` that points to the split directories.

In [ ]:
yaml_cfg = {
    'path': str(DATASET_DIR),
    'train': 'train/images', 'val': 'val/images', 'test': 'test/images',
    'nc': len(CLASS_NAMES), 'names': CLASS_NAMES
}
YAML_PATH = '/content/dataset.yaml'
with open(YAML_PATH, 'w') as f:
    yaml.dump(yaml_cfg, f, default_flow_style=False)
print(open(YAML_PATH).read())

## 6. Experiment Configurations

Four self-contained training runs, each with a specific hypothesis. Run them in order — each is informed by the previous run's results.

| # | Name | Key change |
|---|------|------------|
| 1 | `exp1_rtdetr_x_baseline` | Stronger backbone (RT-DETR-X) |
| 2 | `exp2_rtdetr_l_augmented` | Aggressive augmentation for domain variance |
| 3 | `exp3_rtdetr_l_hires` | Higher resolution (800 px) for small/distant bins |
| 4 | `exp4_rtdetr_l_frozen_backbone` | Freeze backbone, train head only |


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT CONFIGURATIONS
# Each dict is a self-contained training run with a specific hypothesis.
# Run them in order — each one is motivated by the previous run's results.
# ══════════════════════════════════════════════════════════════════════════════

EXPERIMENTS = [

    # ── EXP 1: Stronger backbone — RT-DETR-X ──────────────────────────────────
    # RT-DETR-X has a wider ResNeXt-101 backbone vs RT-DETR-L's ResNet-50.
    # With 600 images we have enough data to avoid the overfitting risk that
    # would make X inferior to L on smaller datasets.
    {
        'name': 'exp1_rtdetr_x_baseline',
        'model': 'rtdetr-x.pt',
        'epochs': 40,
        'imgsz': 640,
        'batch': 8,
        'lr0': 1e-4,        # conservative start; transformer heads are sensitive to LR
        'lrf': 0.01,        # final LR = lr0 * lrf → cosine decay to 1e-6
        'warmup_epochs': 5, # 5-epoch linear warmup before cosine kicks in
        'optimizer': 'AdamW',
        'weight_decay': 1e-4,
        'patience': 20,
        'device': 0,
    },

    # ── EXP 2: RT-DETR-L + aggressive augmentation ─────────────────────────────
    # Moroccan street images have high domain variance (lighting, occlusion,
    # motion blur). Mosaic, MixUp, and perspective transforms directly address
    # this. Copy-paste augmentation synthesises more OVERFLOW/full-bin examples
    # to counter class imbalance without needing more real labels.
    {
        'name': 'exp2_rtdetr_l_augmented',
        'model': 'rtdetr-l.pt',
        'epochs': 40,
        'imgsz': 640,
        'batch': 8,
        'patience': 20,
        'device': 0,
        # Augmentation — ultralytics passes these directly to albumentations
        'mosaic': 1.0,       # always mosaic — fuses 4 images, great for small datasets
        'mixup': 0.15,       # light MixUp — avoids blurring fine bin details
        'copy_paste': 0.2,   # copy minority-class instances into other images
        'degrees': 10.0,     # small rotation — bins are usually upright
        'translate': 0.1,
        'scale': 0.5,        # wide scale jitter — bins appear at very different distances
        'shear': 3.0,
        'perspective': 0.0005, # subtle perspective — dashcam angle variation
        'flipud': 0.0,       # no vertical flip — bins are gravity-bound
        'fliplr': 0.5,
        'hsv_h': 0.02,       # hue shift — handles varied Moroccan lighting
        'hsv_s': 0.7,
        'hsv_v': 0.4,
    },

    # ── EXP 3: Higher resolution ───────────────────────────────────────────────
    # Bins in dashcam footage can be small objects far from the camera.
    # 800px input gives the encoder ~56% more spatial tokens than 640px,
    # directly improving detection of small/distant bins.
    # Trade-off: ~30% more VRAM; use batch=4 to stay safe on T4.
    {
        'name': 'exp3_rtdetr_l_hires',
        'model': 'rtdetr-l.pt',
        'epochs': 40,
        'imgsz': 800,       # ← key change
        'batch': 4,         # reduced to fit VRAM at 800px
        'lr0': 1e-4,
        'lrf': 0.01,
        'warmup_epochs': 5,
        'optimizer': 'AdamW',
        'weight_decay': 1e-4,
        'patience': 20,
        'device': 0,
        'mosaic': 0.5,      # reduce mosaic at high-res to avoid VRAM spikes
        'scale': 0.5,
        'fliplr': 0.5,
        'hsv_s': 0.7,
        'hsv_v': 0.4,
    },

    # ── EXP 4: Freeze backbone, train head only (transfer learning) ────────────
    # RT-DETR pretrained on COCO already knows edges, textures, object shapes.
    # Freezing the ResNet backbone (first 10 layers) and only training the
    # transformer decoder + detection head is more data-efficient — useful
    # when 600 images risks overfitting the backbone to noise in our domain.
    # This also trains ~3× faster per epoch.
    {
        'name': 'exp4_rtdetr_l_frozen_backbone',
        'model': 'rtdetr-l.pt',
        'epochs': 50,      # more epochs because head-only training converges slower
        'imgsz': 640,
        'batch': 8,
        'lr0': 5e-4,        # higher LR safe here — backbone frozen, only head updates
        'lrf': 0.01,
        'warmup_epochs': 3,
        'optimizer': 'AdamW',
        'weight_decay': 1e-4,
        'freeze': 10,       # freeze first 10 layers of ResNet backbone
        'patience': 25,
        'device': 0,
        'mosaic': 1.0,
        'scale': 0.5,
        'fliplr': 0.5,
        'hsv_s': 0.7,
        'hsv_v': 0.4,
        'copy_paste': 0.2,
    },

]

print(f'{len(EXPERIMENTS)} experiments configured.')
for e in EXPERIMENTS:
    print(f"  {e['name']}  —  {e['model']}  imgsz={e['imgsz']}  epochs={e['epochs']}")

## 7. Training & Evaluation

Trains each selected experiment and evaluates on the test split. Change `RUN_EXPERIMENTS` to run a subset (e.g. `[2, 3]`).

In [ ]:
from ultralytics import RTDETR
import shutil

results_log = {}  # name → metrics dict

# ── SELECT WHICH EXPERIMENTS TO RUN ──────────────────────────────────────────
RUN_EXPERIMENTS = [2, 3]   # change to e.g. [0] to run only one

for idx in RUN_EXPERIMENTS:
    cfg = EXPERIMENTS[idx]
    name = cfg['name']
    print(f'\n{"="*60}')
    print(f'  Starting: {name}')
    print(f'{"="*60}')

    model = RTDETR(cfg['model'])

    META_KEYS = {'name', 'model'}
    train_kwargs = {k: v for k, v in cfg.items() if k not in META_KEYS}

    model.train(
        data=YAML_PATH,
        project='/content/runs',
        name=name,          # explicit name passed here, not inside train_kwargs dict
        exist_ok=True,      # allows re-running without incrementing
        verbose=False,
        **train_kwargs
    )

    # Weights land at /content/runs/{name}/weights/best.pt
    weights_path = f'/content/runs/{name}/weights/best.pt'

    # Evaluate on test split — use YOLO() not RTDETR() to avoid path issues
    best = RTDETR(weights_path)
    m = best.val(data=YAML_PATH, split='test', device=0, verbose=False)

    results_log[name] = {
        'mAP50':     round(float(m.box.map50), 4),
        'mAP50-95':  round(float(m.box.map),   4),
        'Precision': round(float(m.box.mp),     4),
        'Recall':    round(float(m.box.mr),     4),
        'model':     cfg['model'],
        'imgsz':     cfg['imgsz'],
        'epochs':    cfg['epochs'],
    }

    print(f"  mAP50={results_log[name]['mAP50']}  "
          f"mAP50-95={results_log[name]['mAP50-95']}  "
          f"P={results_log[name]['Precision']}  "
          f"R={results_log[name]['Recall']}")

print('\n✅ All selected experiments complete.')
